In [0]:
select distinct npi_num__v as hco_npi, corporate_name__v as hco_name_opendata
from com_edp_prd.com_raw.vod_hco
where npi_num__v in (
  '1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364',
  '1093728743', '1184722779', '1861439952'
)

In [0]:
select distinct NPI as hco_npi, ORGANIZATION_NAME as hco_name
from com_edp_prd.com_raw.kom_providers
where PROVIDER_TYPE = 'ORGANIZATION' and npi in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952')

In [0]:
with base AS (
  SELECT DISTINCT
      a.npi_num__v        AS hco_npi,
      a.corporate_name__v AS hco_name,

      -- Hospital parent (lowest parent level in this rollup)
      b.npi_num__v        AS hospital_parent_npi,
      b.corporate_name__v AS hospital_parent_name,

      -- Immediate parent (mid-level)
      c.npi_num__v        AS immediate_parent_npi,
      c.corporate_name__v AS immediate_parent_name,

      -- Top parent (highest-level rollup)
      d.npi_num__v        AS top_parent_npi,
      d.corporate_name__v AS top_parent_name

  FROM com_raw.vod_hco a
  LEFT JOIN com_raw.vod_hco b
      ON a.hospital_parent__v = b.vid__v
  LEFT JOIN com_raw.vod_hco c
      ON a.immediate_parent__v = c.vid__v
  LEFT JOIN com_raw.vod_hco d
      ON a.top_parent__v = d.vid__v

  WHERE a.npi_num__v IN (
      '1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952'
  )
),

/* ---------------------------------------------------------------------------
   STEP 2: Resolve a single parent NPI/name per HCO (best available parent)
   - Preference order:
       1) top parent
       2) immediate parent
       3) hospital parent
       4) null (no parent found)
   - Join the resolved parent fields back onto the hco_360 rows
   --------------------------------------------------------------------------- */
parent_mapping AS (
  SELECT DISTINCT
          hco_npi,
          hco_name,

          /* Parent NPI + Name resolved from SAME LEVEL */
          CASE
              WHEN top_parent_npi IS NOT NULL THEN top_parent_npi
              WHEN immediate_parent_npi IS NOT NULL THEN immediate_parent_npi
              WHEN hospital_parent_npi IS NOT NULL THEN hospital_parent_npi
              ELSE NULL
          END AS parent_npi,

          CASE
              WHEN top_parent_npi IS NOT NULL THEN top_parent_name
              WHEN immediate_parent_npi IS NOT NULL THEN immediate_parent_name
              WHEN hospital_parent_npi IS NOT NULL THEN hospital_parent_name
              ELSE NULL
          END AS parent_name
      FROM base
)

select * from parent_mapping

In [0]:
select distinct npi_num__v, top_parent__v, immediate_parent__v, hospital_parent__v
from com_raw.vod_hco
where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952') and (top_parent__v is not null or hospital_parent__v is not null or immediate_parent__v is not null)

In [0]:
with t1 as (select distinct npi_num__v, top_parent__v, immediate_parent__v, hospital_parent__v
from com_raw.vod_hco
where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952') and (top_parent__v is not null or hospital_parent__v is not null or immediate_parent__v is not null))
select a.npi_num__v as hco_npi, e.corporate_name__v as hco_name, a.top_parent__v, b.corporate_name__v as top_parent_name,
a.immediate_parent__v, c.corporate_name__v as immediate_parent_name,
a.hospital_parent__v, d.corporate_name__v as hospital_parent_name
from t1 as a
left join com_raw.vod_hco as b on a.top_parent__v = b.vid__v
left join com_raw.vod_hco as c on a.immediate_parent__v = c.vid__v
left join com_raw.vod_hco as d on a.hospital_parent__v = d.vid__v
left join com_raw.vod_hco as e on a.npi_num__v = e.npi_num__v

In [0]:
WITH t1 AS (
    SELECT DISTINCT 
        npi_num__v, 
        vid__v,
        top_parent__v, 
        immediate_parent__v, 
        hospital_parent__v 
    FROM com_raw.vod_hco 
    WHERE 
        top_parent__v = '931278329388861343'
        OR immediate_parent__v = '931278329388861343'
        OR hospital_parent__v = '931278329388861343'
)
SELECT 
    a.vid__v as child_vid,
    a.npi_num__v AS child_hco_npi,
    e.corporate_name__v AS child_hco_name,
    a.top_parent__v,
    b.corporate_name__v AS top_parent_name,
    a.immediate_parent__v,
    c.corporate_name__v AS immediate_parent_name,
    a.hospital_parent__v,
    d.corporate_name__v AS hospital_parent_name
FROM t1 AS a
LEFT JOIN com_raw.vod_hco AS b ON a.top_parent__v = b.vid__v
LEFT JOIN com_raw.vod_hco AS c ON a.immediate_parent__v = c.vid__v
LEFT JOIN com_raw.vod_hco AS d ON a.hospital_parent__v = d.vid__v
LEFT JOIN com_raw.vod_hco AS e ON a.npi_num__v = e.npi_num__v

### Parent Level Info (16th Jan)

In [0]:
WITH target_top_parents AS (
    SELECT DISTINCT top_parent__v
    FROM com_raw.vod_hco
    WHERE npi_num__v IN ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364')
      AND top_parent__v IS NOT NULL
),

top_parent_child AS (
    SELECT
        tp.top_parent__v AS top_parent_vid,
        p.corporate_name__v AS top_parent_name,
        p.npi_num__v AS top_parent_npi,
        e.name AS top_parent_type,
        c.vid__v AS child_vid,
        c.corporate_name__v AS child_name,
        c.npi_num__v AS child_npi,
        d.name AS child_type,
        c.hco_status__v AS child_status
    FROM target_top_parents tp
    JOIN com_raw.vod_hco p
        ON tp.top_parent__v = p.vid__v
    JOIN com_raw.vod_hco c
        ON (tp.top_parent__v = c.top_parent__v
         OR tp.top_parent__v = c.immediate_parent__v
         OR tp.top_parent__v = c.hospital_parent__v)
         and c.npi_num__v in (select distinct hco_npi_old from com_edp_prd.cmpa_insights_internal_schema.reference_file)
    LEFT JOIN com_raw.vod_references d
        ON c.hco_type__v = d.code AND d.reference_type = 'HCOType'
    LEFT JOIN com_raw.vod_references e
        ON p.hco_type__v = e.code AND e.reference_type = 'HCOType'
    ORDER BY top_parent_vid, child_vid
),

appending_tier_flag AS (
  SELECT *,
    CASE
      WHEN child_npi IN ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364')
      THEN 1 ELSE 0
    END AS tier1_flag
  FROM top_parent_child
),

child_count AS (
    SELECT top_parent_vid, COUNT(DISTINCT child_vid) AS no_child_accounts
    FROM appending_tier_flag
    GROUP BY top_parent_vid
),

appending_child_count AS (
  SELECT a.*, b.no_child_accounts
  FROM appending_tier_flag a
  LEFT JOIN child_count b
    ON a.top_parent_vid = b.top_parent_vid
),

/* -------------------------------------------------------------------------
   NEW: Tier1-only base_table built from your result (no hard-coded VALUES)
   ------------------------------------------------------------------------- */
base_table AS (
  SELECT DISTINCT TRY_CAST(child_npi AS STRING) AS hco_npi
  FROM appending_child_count
  WHERE tier1_flag = 1
    AND child_npi IS NOT NULL
),

hco_engaged AS (
  SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
  FROM com_edp_prd.com_raw.vcrm_call2__v a
  LEFT JOIN com_intgr.customer b
    ON a.account__v = b.id
  WHERE TRY_CAST(b.npi__v AS STRING) IN (SELECT hco_npi FROM base_table)
    AND a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
),

hco_profiled AS (
  SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
  FROM com_intgr.survey_target a
  LEFT JOIN com_intgr.customer b
    ON a.account__v = b.id
  WHERE TRY_CAST(b.npi__v AS STRING) IN (SELECT hco_npi FROM base_table)
),

tier1_flags AS (
  SELECT
    bt.hco_npi,
    CASE WHEN he.npi IS NOT NULL THEN 1 ELSE 0 END AS is_hco_engaged,
    CASE WHEN hp.npi IS NOT NULL THEN 1 ELSE 0 END AS is_hco_profiled
  FROM base_table bt
  LEFT JOIN hco_engaged he
    ON bt.hco_npi = he.npi
  LEFT JOIN hco_profiled hp
    ON bt.hco_npi = hp.npi
),

/* ------------------- your existing address/territory pipeline ------------------- */
target_vids AS (
    SELECT DISTINCT child_vid AS hco_vid
    FROM appending_child_count
    WHERE child_vid IS NOT NULL
    UNION
    SELECT DISTINCT top_parent_vid AS hco_vid
    FROM appending_child_count
    WHERE top_parent_vid IS NOT NULL
),

hco_address_latest AS (
    SELECT
        b.entity_vid__v AS hco_vid,
        b.address_line_1__v AS address_line_1,
        b.postal_code_cda__v AS postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY b.entity_vid__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_address b
    JOIN target_vids t
        ON t.hco_vid = b.entity_vid__v
    WHERE b.entity_type__v = 'HCO'
      AND b.record_state__v = 'VALID'
      AND b.address_status__v IN ('A','DS')
      AND b.address_verification_status__v NOT IN ('NS','U')
),

hco_address_latest_1 AS (
    SELECT hco_vid, address_line_1, postal_code
    FROM hco_address_latest
    WHERE rn = 1
),

addresses_child_top_parent AS (
    SELECT
        a.*,
        caddr.address_line_1 AS child_address,
        caddr.postal_code    AS child_zip,
        paddr.address_line_1 AS top_parent_address,
        paddr.postal_code    AS top_parent_zip
    FROM appending_child_count a
    LEFT JOIN hco_address_latest_1 caddr
        ON a.child_vid = caddr.hco_vid
    LEFT JOIN hco_address_latest_1 paddr
        ON a.top_parent_vid = paddr.hco_vid
),

territory_region_child AS (
    SELECT
        a.*,
        COALESCE(z.territory_name, '-') AS child_territory,
        COALESCE(z.region_name, '-')    AS child_region,
        COALESCE(z.city, '-')           AS child_city,
        COALESCE(z.state, '-')          AS child_state
    FROM addresses_child_top_parent a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON TRY_CAST(NULLIF(a.child_zip, '-') AS BIGINT) = z.zipcode
),

territory_region_parent AS (
    SELECT
        a.*,
        COALESCE(z.territory_name, '-') AS top_parent_territory,
        COALESCE(z.region_name, '-')    AS top_parent_region,
        COALESCE(z.city, '-')           AS top_parent_city,
        COALESCE(z.state, '-')          AS top_parent_state
    FROM territory_region_child a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON TRY_CAST(NULLIF(a.top_parent_zip, '-') AS BIGINT) = z.zipcode
),

/* -------------------------------------------------------------------------
   FINAL: attach engaged/profiled to CHILD NPI; NA when tier1_flag = 0
   ------------------------------------------------------------------------- */
final as (SELECT
  trp.*,

  CASE
    WHEN trp.tier1_flag = 0 THEN 'Not applicable'
    WHEN tf.is_hco_engaged = 1 THEN '1'
    ELSE '0'
  END AS is_hco_engaged,

  CASE
    WHEN trp.tier1_flag = 0 THEN 'Not applicable'
    WHEN tf.is_hco_profiled = 1 THEN '1'
    ELSE '0'
  END AS is_hco_profiled

FROM territory_region_parent trp
LEFT JOIN tier1_flags tf
  ON TRY_CAST(trp.child_npi AS STRING) = tf.hco_npi)

select * from final

In [0]:
WITH base_npis AS (
    select distinct hco_npi_old as npi
    from cmpa_insights_internal_schema.reference_file
),

hco_engaged AS (
    SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
    FROM com_edp_prd.com_raw.vcrm_call2__v a
    JOIN com_intgr.customer b
        ON a.account__v = b.id
    WHERE a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
),

hco_profiled AS (
    SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
    FROM com_intgr.survey_target a
    JOIN com_intgr.customer b
        ON a.account__v = b.id
)

SELECT
    b.npi,
    CASE WHEN e.npi IS NOT NULL THEN 1 ELSE 0 END AS is_engaged,
    CASE WHEN p.npi IS NOT NULL THEN 1 ELSE 0 END AS is_profiled
FROM base_npis b
LEFT JOIN hco_engaged e
    ON b.npi = e.npi
LEFT JOIN hco_profiled p
    ON b.npi = p.npi
ORDER BY b.npi;


In [0]:
SELECT DISTINCT
    b.npi__v AS npi, a.entity_display_name__v as name_from_crm_calls_table
  FROM com_edp_prd.com_raw.vcrm_call2__v AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  and a.call_date__v between '2025-07-01' and '2025-12-31'
  where b.npi__v is not null

In [0]:
with t1 as (select distinct vid__v ,npi_num__v, top_parent__v, immediate_parent__v, hospital_parent__v
from com_raw.vod_hco
where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364') and (top_parent__v is not null or hospital_parent__v is not null or immediate_parent__v is not null)),

base_table as (
  select *, corporate_name__v
  from com_raw.vod_hco
  where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364')
)
select distinct vid__v, base_table.corporate_name__v from base_table

In [0]:
with t1 as (select distinct vid__v ,npi_num__v, top_parent__v, immediate_parent__v, hospital_parent__v
from com_raw.vod_hco
where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364') and (top_parent__v is not null or hospital_parent__v is not null or immediate_parent__v is not null)),
t2 as (select a.vid__v as hco_vid, a.npi_num__v as hco_npi, e.corporate_name__v as hco_name, 
a.top_parent__v, b.corporate_name__v as top_parent_name,
a.immediate_parent__v, c.corporate_name__v as immediate_parent_name,
a.hospital_parent__v, d.corporate_name__v as hospital_parent_name
from t1 as a
left join com_raw.vod_hco as b on a.top_parent__v = b.vid__v
left join com_raw.vod_hco as c on a.immediate_parent__v = c.vid__v
left join com_raw.vod_hco as d on a.hospital_parent__v = d.vid__v
left join com_raw.vod_hco as e on a.npi_num__v = e.npi_num__v),
t3 as (
  select distinct *
  from com_raw.vod_hco
  where top_parent__v in (select distinct top_parent__v from t2)
  or immediate_parent__v in (select distinct top_parent__v from t2)
  or hospital_parent__v in (select distinct top_parent__v from t2)
)
select * from t1

In [0]:
WITH target_hcos AS (
    SELECT '1437365186' AS hco_npi UNION ALL
    SELECT '1235234535' UNION ALL
    SELECT '1295789907' UNION ALL
    SELECT '1053632463' UNION ALL
    SELECT '1114969169' UNION ALL
    SELECT '1104819366' UNION ALL
    SELECT '1346297843' UNION ALL
    SELECT '1336245828' UNION ALL
    SELECT '1003102781' UNION ALL
    SELECT '1164686879' UNION ALL
    SELECT '1215921457' UNION ALL
    SELECT '1164426896' UNION ALL
    SELECT '1750482022' UNION ALL
    SELECT '1366515488' UNION ALL
    SELECT '1912939703' UNION ALL
    SELECT '1205935012' UNION ALL
    SELECT '1194787218' UNION ALL
    SELECT '1548212988' UNION ALL
    SELECT '1043447253' UNION ALL
    SELECT '1891765178' UNION ALL
    SELECT '1023105400' UNION ALL
    SELECT '1649347469' UNION ALL
    SELECT '1336495910' UNION ALL
    SELECT '1760476659' UNION ALL
    SELECT '1235339227' UNION ALL
    SELECT '1649261462' UNION ALL
    SELECT '1184649345' UNION ALL
    SELECT '1144266024' UNION ALL
    SELECT '1578693321' UNION ALL
    SELECT '1013143213' UNION ALL
    SELECT '1013062769' UNION ALL
    SELECT '1073053757' UNION ALL
    SELECT '1285832634' UNION ALL
    SELECT '1285174649' UNION ALL
    SELECT '1659877280' UNION ALL
    SELECT '1003063280' UNION ALL
    SELECT '1366556227' UNION ALL
    SELECT '1275564098' UNION ALL
    SELECT '1093894131' UNION ALL
    SELECT '1013924372' UNION ALL
    SELECT '1225249865' UNION ALL
    SELECT '1609824010' UNION ALL
    SELECT '1669429577' UNION ALL
    SELECT '1700128592' UNION ALL
    SELECT '1033439732' UNION ALL
    SELECT '1760480503' UNION ALL
    SELECT '1235148594' UNION ALL
    SELECT '1750458485' UNION ALL
    SELECT '1083949382' UNION ALL
    SELECT '1114924834' UNION ALL
    SELECT '1083789630' UNION ALL
    SELECT '1669462420' UNION ALL
    SELECT '1093808040' UNION ALL
    SELECT '1477643690' UNION ALL
    SELECT '1023188851' UNION ALL
    SELECT '1477549756' UNION ALL
    SELECT '1154302727' UNION ALL
    SELECT '1003961251' UNION ALL
    SELECT '1235214834' UNION ALL
    SELECT '1083630073' UNION ALL
    SELECT '1265694442' UNION ALL
    SELECT '1376544320' UNION ALL
    SELECT '1467525790' UNION ALL
    SELECT '1679973364' UNION ALL
    SELECT '1568596765' UNION ALL
    SELECT '1003878539' UNION ALL
    SELECT '1144548322' UNION ALL
    SELECT '1689747552' UNION ALL
    SELECT '1326092404' UNION ALL
    SELECT '1639370059' UNION ALL
    SELECT '1063702785' UNION ALL
    SELECT '1235582925' UNION ALL
    SELECT '1598784555' UNION ALL
    SELECT '1275694184' UNION ALL
    SELECT '1396882205' UNION ALL
    SELECT '1669683512' UNION ALL
    SELECT '1932280666' UNION ALL
    SELECT '1013924182'
),
vod as (
  select a.*, b.top_parent__v, b.immediate_parent__v, b.hospital_parent__v, b.vid__v
  from target_hcos as a
  left join com_raw.vod_hco as b on a.hco_npi = b.npi_num__v
)
select distinct hco_npi
from vod
where (vid__v is null) or ((top_parent__v is null) and (immediate_parent__v is null) and (hospital_parent__v is null))

### Pulling Tx Counts, Dx Counts, All patient counts for the Tier1 HCOs

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical Events - Dx (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - Dx (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (5yr) - CORRECTED NPI LOGIC
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical Events - NDC codes (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Medical Events - Procedure codes (RENDERING_NPI only)
SELECT DISTINCT 
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743', 
                         'S9357', 'S9379', '38206', '38230', '38232', 
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - NDC codes (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 3: TREATMENT CLAIMS FOR ELIGIBILITY (2yr)
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
-- =============================================================================

-- Specified: 2+ E761 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Specified Patients: 2+ E761 Dx + any Tx in 2yr
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;


-- Incremental: 2+ E763 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Elaprase Tx in 2yr (for incremental eligibility)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');


-- Incremental Patients: 2+ E763 Dx + Elaprase Tx in 2yr + NOT specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);


-- All Eligible Patients
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

-- Dx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

-- Tx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
create or replace temporary view all_patients_non_unique as
with hcp_level as (
  select * from all_patient_claims
),
hco_level as (
  select a.*, b.hco_npi
  from hcp_level as a
  left join cmpa_insights_internal_schema.reference_file_0109 as b on a.npi = b.hcp_npi
)
select hco_npi, count(distinct patient_id) as all_patients_non_unique_count
from hco_level
where npi is not null and hco_npi != '-'
group by 1 order by 2 desc

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_patients_unique AS
WITH hcp_metrics AS (
    SELECT 
        a.PATIENT_ID,
        a.NPI,
        
        -- Specialty Classification
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 'NPPA'
            WHEN a.NPI IS NULL 
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,
        
        -- Priority (Tier 1)
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 5
            WHEN a.NPI IS NULL 
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,
        
        -- Visit Count: Dx + Tx combined (Tier 2)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,
        
        -- Dx-only visit count (for reference)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        
        -- Tx-only visit count (for reference)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,
        
        -- Most Recent Visit (Tier 3)
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT
        
    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p 
        ON a.NPI = p.NPI
    GROUP BY 
        a.PATIENT_ID, 
        a.NPI,
        p.primary_specialty, 
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT 
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY 
                SPECIALTY_PRIORITY ASC,    -- Tier 1: Specialty (lower = better)
                NO_OF_VISITS DESC,         -- Tier 2: Total visits (higher = better)
                MOST_RECENT_VISIT DESC,    -- Tier 3: Recency (more recent = better)
                NPI ASC                    -- Tier 4: NPI tiebreaker (lower wins)
        ) AS HCP_RANK
    FROM hcp_metrics
),
hcp_level as (
  SELECT 
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1
),
hco_level as (
  select a.*, b.hco_npi
  from hcp_level as a
  left join cmpa_insights_internal_schema.reference_file_0109 as b on a.primary_hcp_npi = b.hcp_npi
)
-- select hco_npi, count(distinct patient_id) as all_patiets_unique_count
-- from hco_level
-- where primary_hcp_npi is not null and hco_npi != '-'
-- group by 1 order by 2 desc
select * from hcp_level

In [0]:
-- create or replace temporary view all_dx_non_unique as
with hcp_level as (
  select * from all_dx_claims
),
hco_level as (
  select a.*, b.hco_npi
  from hcp_level as a
  left join cmpa_insights_internal_schema.reference_file_0109 as b on a.npi = b.hcp_npi
)
select hco_npi, count(distinct patient_id) as all_dx_non_unique_count
from hco_level
where npi is not null and hco_npi != '-'
group by 1 order by 2 desc

In [0]:
-- CREATE OR REPLACE TEMPORARY VIEW all_dx_unique AS
WITH hcp_metrics AS (
    SELECT 
        a.PATIENT_ID,
        a.NPI,
        
        -- Specialty Classification
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 'NPPA'
            WHEN a.NPI IS NULL 
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,
        
        -- Priority (Tier 1)
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 5
            WHEN a.NPI IS NULL 
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,
        
        -- Visit Count: Dx + Tx combined (Tier 2)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,
        
        -- Dx-only visit count (for reference)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        
        -- Tx-only visit count (for reference)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,
        
        -- Most Recent Visit (Tier 3)
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT
        
    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p 
        ON a.NPI = p.NPI
    GROUP BY 
        a.PATIENT_ID, 
        a.NPI,
        p.primary_specialty, 
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT 
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY 
                SPECIALTY_PRIORITY ASC,    -- Tier 1: Specialty (lower = better)
                NO_OF_VISITS DESC,         -- Tier 2: Total visits (higher = better)
                MOST_RECENT_VISIT DESC,    -- Tier 3: Recency (more recent = better)
                NPI ASC                    -- Tier 4: NPI tiebreaker (lower wins)
        ) AS HCP_RANK
    FROM hcp_metrics
),
hcp_level as (
  SELECT 
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1
),
hco_level as (
  select a.*, b.hco_npi
  from hcp_level as a
  left join cmpa_insights_internal_schema.reference_file_0109 as b on a.primary_hcp_npi = b.hcp_npi
)
-- select hco_npi, count(distinct patient_id) as all_patiets_unique_count
-- from hco_level
-- where primary_hcp_npi is not null and hco_npi != '-'
-- group by 1 order by 2 desc
select * from hcp_level

## Appendix

In [0]:
WITH target_top_parents AS (
    SELECT DISTINCT top_parent__v
    FROM com_raw.vod_hco
    WHERE npi_num__v IN ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364')
    AND top_parent__v IS NOT NULL
),
top_parent_child AS (
    SELECT tp.top_parent__v AS top_parent_vid, p.corporate_name__v AS top_parent_name, p.npi_num__v AS top_parent_npi, e.name as top_parent_type, c.vid__v AS child_vid, c.corporate_name__v AS child_name, c.npi_num__v AS child_npi, 
    d.name as child_type,
    -- e.name as child_class_of_trade, 
    c.hco_status__v as child_status
    FROM target_top_parents tp
    JOIN com_raw.vod_hco p ON tp.top_parent__v = p.vid__v
    JOIN com_raw.vod_hco c ON (tp.top_parent__v = c.top_parent__v or tp.top_parent__v = c.immediate_parent__v or tp.top_parent__v = c.hospital_parent__v)
    left join com_raw.vod_references as d on c.hco_type__v = d.code and d.reference_type = 'HCOType'
    left join com_raw.vod_references as e on p.hco_type__v = e.code and e.reference_type = 'HCOType'
    ORDER BY top_parent_vid, child_vid
),
appending_tier_flag as (
  select *,
  case when child_npi in ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364') then 1 else 0 end as tier1_flag
  from top_parent_child
),
child_count AS (
    SELECT top_parent_vid, COUNT(DISTINCT child_vid) AS no_child_accounts
    FROM appending_tier_flag
    GROUP BY top_parent_vid
    ORDER BY no_child_accounts DESC
),
appending_child_count as (
  select a.*, b.no_child_accounts
  from appending_tier_flag as a
  left join child_count as b on a.top_parent_vid = b.top_parent_vid
),

-- Start from your existing CTE: appending_child_count

target_vids AS (
    -- collect all relevant HCO VIDs (child + top parent) so we only scan addresses we need
    SELECT DISTINCT child_vid AS hco_vid
    FROM appending_child_count
    WHERE child_vid IS NOT NULL

    UNION

    SELECT DISTINCT top_parent_vid AS hco_vid
    FROM appending_child_count
    WHERE top_parent_vid IS NOT NULL
)

, hco_address_latest AS (
    SELECT
        b.entity_vid__v                   AS hco_vid,
        b.address_line_1__v               AS address_line_1,
        b.postal_code_cda__v              AS postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY b.entity_vid__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_address b
    JOIN target_vids t
        ON t.hco_vid = b.entity_vid__v
    WHERE b.entity_type__v = 'HCO'
      AND b.record_state__v = 'VALID'
      AND b.address_status__v IN ('A','DS')
      AND b.address_verification_status__v NOT IN ('NS','U')
)

, hco_address_latest_1 AS (
    SELECT
        hco_vid,
        address_line_1,
        postal_code
    FROM hco_address_latest
    WHERE rn = 1
)

, addresses_child_top_parent AS (
    SELECT
        a.*,

        -- CHILD address fields
        caddr.address_line_1 AS child_address,
        caddr.postal_code    AS child_zip,

        -- TOP PARENT address fields
        paddr.address_line_1 AS top_parent_address,
        paddr.postal_code    AS top_parent_zip

    FROM appending_child_count a
    LEFT JOIN hco_address_latest_1 caddr
        ON a.child_vid = caddr.hco_vid
    LEFT JOIN hco_address_latest_1 paddr
        ON a.top_parent_vid = paddr.hco_vid
)

, territory_region_child AS (
    SELECT
        a.*,
        COALESCE(z.territory_name, '-') AS child_territory,
        COALESCE(z.region_name, '-')    AS child_region,
        COALESCE(z.city, '-')           AS child_city,
        COALESCE(z.state, '-')          AS child_state
    FROM addresses_child_top_parent a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON TRY_CAST(NULLIF(a.child_zip, '-') AS BIGINT) = z.zipcode
)

, territory_region_parent AS (
    SELECT
        a.*,
        COALESCE(z.territory_name, '-') AS top_parent_territory,
        COALESCE(z.region_name, '-')    AS top_parent_region,
        COALESCE(z.city, '-')           AS top_parent_city,
        COALESCE(z.state, '-')          AS top_parent_state
    FROM territory_region_child a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON TRY_CAST(NULLIF(a.top_parent_zip, '-') AS BIGINT) = z.zipcode
)

SELECT *
FROM territory_region_parent;


In [0]:
WITH target_top_parents AS (
    SELECT DISTINCT top_parent__v
    FROM com_raw.vod_hco
    WHERE npi_num__v IN ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364')
    AND top_parent__v IS NOT NULL
),
top_parent_child AS (
    SELECT tp.top_parent__v AS top_parent_vid, p.corporate_name__v AS top_parent_name, p.npi_num__v AS top_parent_npi, c.vid__v AS child_vid, c.corporate_name__v AS child_name, c.npi_num__v AS child_npi, c.hco_type__v as child_type,
    e.name as child_class_of_trade, c.hco_status__v as child_status
    FROM target_top_parents tp
    JOIN com_raw.vod_hco p ON tp.top_parent__v = p.vid__v
    JOIN com_raw.vod_hco c ON (tp.top_parent__v = c.top_parent__v or tp.top_parent__v = c.immediate_parent__v or tp.top_parent__v = c.hospital_parent__v)
    left join com_raw.vod_references as d on c.hco_type__v = d.code and d.reference_type = 'HCOType'
    left join com_raw.vod_references as e on c.major_class_of_trade__v = e.code and e.reference_type = 'MajorClassofTrade'
    ORDER BY top_parent_vid, child_vid
),
tier_flag as (
  select *,
  case when child_npi in ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364') then 1 else 0 end as tier1_flag
  from top_parent_child
),
child_count AS (
    SELECT top_parent_vid, COUNT(DISTINCT child_vid) AS no_child_accounts
    FROM tier_flag
    where child_type = '4:6'
    -- where child_type in ('Organization, Group Practice', 'Organization, PHS Outpatient', 'Organization, Dept at Hospital', 'Imaging Center', 'Organization, Group at Hospital', 'Pharmacy, Retail', 'Education Establishment', 'Organization, Other Org', 'Transplant Center', 'Organization, Hospital', 'Walk-In Clinic', 'Organization, Lab', 'Organization, Admin Only', 'Pharmacy, Hospital', 'Dialysis & Infusion Centers', 'Organization, Nursing Home / Long Term Care', 'Residency Program, Residency', 'Organization, Dentistry Group', 'Organization, CMS Teaching Hospital', 'Dialysis & Infusion Centers, Hospital-based Dialysis/Infusion', 'Psychotherapy / Counselling Center', 'Ambulatory Surgery Center', 'Home Health Care Agency', 'Pharmacy, Specialty Pharmacy', 'Organization, Cystic Fibrosis Center', 'Organization, Hospice', 'Optical Center', 'Organization, Community MH Center', 'Physiotherapy Center', 'Organization, Health System', 'Pharmacy, Other', 'Medical School, General', 'Organization, HMO/Insurance', 'Organization, Podiatry Group', 'Organization, Hemophilia Center', 'Pharmacy, Home Health', 'Distributor, Medical Equipment', 'Organization, Accountable Care Organizations (ACOs)', 'Organization, Multiple Sclerosis Center', 'Medical School, Other', 'Organization, Poison Control Center', 'Pharmacy, Mail Order Pharmacy', 'Organization, Veterinary Group', 'Pharmacy, Compounding Pharmacy', 'Organization, Residential Facility', 'Distributor, General', 'Organization, Occupational Therapy Group', 'Pharmacy, Oncology Specialty Pharmacy', 'Organization, Chiropractic Group', 'Healthcare Manufacturer, General', 'Pharmacy, Long Term Care Pharmacy Provider', 'Pharmacy, Other Specialty Clinic')
    GROUP BY top_parent_vid
    ORDER BY no_child_accounts DESC
)
select * from child_count

In [0]:
select distinct *
from com_intgr.customer

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file_0109

In [0]:
-- =============================================================================
-- Primary HCP Assignment (Dx + Tx Claims) - CORRECTED
-- =============================================================================
-- Diagnosis Window: 2020-08-01 to 2025-11-30 (5 years)
-- Treatment Window: 2023-08-01 to 2025-11-30 (2 years)
-- 
-- Primary HCP considers BOTH Dx and Tx claims for visit counting
-- 
-- NPI LOGIC (aligned with GTM file):
--   - NDC (Medical): COALESCE(RENDERING_NPI, REFERRING_NPI)
--   - Procedure (Medical): RENDERING_NPI only
--   - Pharmacy: PRESCRIBER_NPI
-- =============================================================================


-- =============================================================================
-- STEP 1: DIAGNOSIS CLAIMS (5yr)
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical Events - Dx (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - Dx (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (5yr) - CORRECTED NPI LOGIC
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical Events - NDC codes (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Medical Events - Procedure codes (RENDERING_NPI only)
SELECT DISTINCT 
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743', 
                         'S9357', 'S9379', '38206', '38230', '38232', 
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - NDC codes (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 3: TREATMENT CLAIMS FOR ELIGIBILITY (2yr)
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
-- =============================================================================

-- Specified: 2+ E761 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Specified Patients: 2+ E761 Dx + any Tx in 2yr
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;


-- Incremental: 2+ E763 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Elaprase Tx in 2yr (for incremental eligibility)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');


-- Incremental Patients: 2+ E763 Dx + Elaprase Tx in 2yr + NOT specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);


-- All Eligible Patients
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

-- Dx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

-- Tx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);


-- =============================================================================
-- STEP 6: PRIMARY HCP ASSIGNMENT WITH 4-TIER RANKING
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT 
        a.PATIENT_ID,
        a.NPI,
        
        -- Specialty Classification
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 'NPPA'
            WHEN a.NPI IS NULL 
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,
        
        -- Priority (Tier 1)
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 5
            WHEN a.NPI IS NULL 
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,
        
        -- Visit Count: Dx + Tx combined (Tier 2)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,
        
        -- Dx-only visit count (for reference)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        
        -- Tx-only visit count (for reference)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,
        
        -- Most Recent Visit (Tier 3)
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT
        
    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p 
        ON a.NPI = p.NPI
    GROUP BY 
        a.PATIENT_ID, 
        a.NPI,
        p.primary_specialty, 
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT 
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY 
                SPECIALTY_PRIORITY ASC,    -- Tier 1: Specialty (lower = better)
                NO_OF_VISITS DESC,         -- Tier 2: Total visits (higher = better)
                MOST_RECENT_VISIT DESC,    -- Tier 3: Recency (more recent = better)
                NPI ASC                    -- Tier 4: NPI tiebreaker (lower wins)
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT 
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;

-- -- Creating a table for Primary HCP
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
-- SELECT * FROM primary_hcp_assignment;

In [0]:
select * from primary_hcp

In [0]:
select primary_hcp_npi, count(distinct patient_id)
from primary_hcp
group by 1 order by 2 desc

In [0]:
-- select *
-- from cmpa_insights_internal_schema.reference_file_0109 limit 1

select * except (hcp_primary_specialty), hcp_primary_specialty as hcp_specialty
from cmpa_insights_internal_schema.reference_file_0109
limit 1

In [0]:
select * from cmpa_insights_internal_schema.reference_file limit 1

In [0]:
select *
from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping

In [0]:
with t1 as (select a.*, b.territory_id, c.region_id
from cmpa_insights_internal_schema.reference_file_0109 as a
left join (select distinct territory_id, territory_name
from cmpa_insights_internal_schema.zip_to_territory_mapping) as b on a.territory = b.territory_name
left join (select distinct region_id, region_name
from cmpa_insights_internal_schema.zip_to_territory_mapping) as c on a.region = c.region_name)
select * except (hcp_primary_specialty), hcp_primary_specialty as hcp_specialty
from t1

In [0]:
SELECT 
  * EXCEPT (hcp_primary_specialty), 
  hcp_primary_specialty AS hcp_specialty
FROM (
  SELECT 
    a.*, 
    b.territory_id, 
    c.region_id
  FROM cmpa_insights_internal_schema.reference_file_0109 AS a
  LEFT JOIN (
    SELECT DISTINCT 
      territory_id, 
      territory_name
    FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) AS b 
    ON a.territory = b.territory_name
  LEFT JOIN (
    SELECT DISTINCT 
      region_id, 
      region_name
    FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) AS c 
    ON a.region = c.region_name
)

In [0]:
select distinct territory_id, territory_name
from cmpa_insights_internal_schema.zip_to_territory_mapping
order by territory_name

In [0]:
select distinct region_id, region_name
from cmpa_insights_internal_schema.zip_to_territory_mapping
order by region_name

In [0]:
select * from cmpa_insights_internal_schema.zip_to_territory_mapping

In [0]:
select distinct territory_name from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
WHERE region_name = 'Southern Cal'

In [0]:
UPDATE com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
SET territory_id = 2010
WHERE territory_name = 'Southern Cal';

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360

### QC

In [0]:
with all_claims as (
  SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Dx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Dx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761','E763')
  AND TRANSACTION_STATUS = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001','540920700')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Tx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001','540920700')
  AND TRANSACTION_RESULT = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                          '38206','38230','38232','38240','38241','38242','38243','38250')
),
relevant_patients as (
  select *
  from all_claims
  where patient_id in ('WYB4NG8H')
),
patient_geography AS (
  SELECT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
          valid_to_date DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_patient_geography
  )
  WHERE rn = 1
),
pulling_relevant_info as (
  select a.patient_id, a.npi, a.fill_date, b.PATIENT_YOB, (year(current_date) - year(b.PATIENT_YOB)) as age, c.PATIENT_STATE, d.FIRST_NAME, d.LAST_NAME, concat(d.FIRST_NAME, ' ', d.LAST_NAME) as hcp_name, d.PRIMARY_SPECIALTY, d.SECONDARY_SPECIALTY, a.claim_type, e.PAYER_NAME, e.INSURANCE_GROUP
  from relevant_patients as a
  left join com_edp_prd.com_raw.kom_patient_demographics as b on a.patient_id = b.PATIENT_ID
  left join patient_geography as c on a.patient_id = c.PATIENT_ID
  left join com_edp_prd.com_raw.kom_providers as d on a.npi = d.npi and d.provider_type = 'INDIVIDUAL'
  left join com_edp_prd.com_raw.kom_plans as e on a.plan_id = e.KH_PLAN_ID
)
select * from pulling_relevant_info